# variant-files.ipynb
GOAL: Identify all somatic and germline variant call files (.vcf) corresponding to tumors in our WGS cohort.

### DNANexus token
DNANexus token docs: https://documentation.dnanexus.com/user/login-and-logout#using-tokens  
tl;dr create a token, then login using   
`dx login --token [token]`  
to update your configuration in ~/.dnanexus_config/environment.json.

In [ ]:
import pandas as pd
import sys
from pathlib import Path
import dxpy
import os
import json

sys.path.append('../../src')
Path("out").mkdir(parents=True, exist_ok=True)

import data_imports

pd.set_option('display.max_columns', None)

dxpy.api.system_whoami()

In [ ]:
ppce_id='project-Fz9yBjQ9fj2QPbFY16B8JG8X'
gvcf_paths = ['/restricted/gVCF/','/Data Requested 2022-09-06/gVCF/']
svcf_paths = ['/restricted/Somatic_VCF','/Data Requested 2022-09-06/Somatic_VCF']

In [ ]:
def get_dx_files(project_id,path):
    gen = dxpy.find_data_objects(project=project_id, folder=path, recurse=False, describe=True)
    file_list = set()
    for file_obj in gen:
        file_name = file_obj['describe']['name']
        file_id = file_obj['describe']['id']
        file_list.add((file_name,file_id))
    return file_list

In [ ]:
PROJECT_ROOT='../..'
SAMPLE_INFO_DIR='data/source/sjcloud/sample_info'
f1=Path(PROJECT_ROOT,SAMPLE_INFO_DIR,'2022-09-08_SAMPLE_INFO.txt') # 23979 vcfs
f2=Path(PROJECT_ROOT,SAMPLE_INFO_DIR,'SAMPLE_INFO_2022-09-06.txt') # 23979 vcfs. Same as F1?
f3=Path(PROJECT_ROOT,SAMPLE_INFO_DIR,'SAMPLE_INFO PedPanCancer ecDNA.txt') # 15280 vcfs
f4=Path(PROJECT_ROOT,SAMPLE_INFO_DIR,'SAMPLE_INFO_2022-03-02.tsv') # 45065 vcfs
f5=Path(PROJECT_ROOT,SAMPLE_INFO_DIR,'SAMPLE_INFO_batch_2022-03-03.txt') # 0 vcfs

In [ ]:
def read_sample_info(path):
    path=str(path.resolve())
    df = pd.read_csv(path,sep='\t')
    df['sample_type'] = df.sample_type.str.lower()
    df['file_name'] = df.file_path.map(lambda x: x.split('/')[-1])
    return df
def get_vcfs(path):
    df = read_sample_info(path)
    df = df[df.file_type.isin(['gVCF','Somatic_VCF'])]
    return df
def text_venn2(s1, s2):
    print(f'size of set 1: {len(s1)}')
    print(f'size of set 2: {len(s2)}')
    print(f'Samples in s1 not in s2: {len(s1 - s2)}')
    print(f'Samples in s2 not in s1: {len(s2 - s1)}')
    print(f'Overlap: {len(s1 & s2)}')
def flatten(df,index_col):
    # helper function for get_germline_files, get_somatic_files
    df = df.sort_values([index_col, 'file_name']).reset_index(drop=True)
    df['ct'] = df.groupby(index_col).cumcount()
    pivot_cols = ['file_name', 'file_id']
    t = df.pivot(index=index_col,
                 columns='ct',
                 values=pivot_cols)
    filetypes = ['vcf','tbi']
    t.columns = [f'{filetypes[i]}_{col}' for col, i in t.columns]
    df = df.drop(columns=['file_name', 'file_id', 'ct']).drop_duplicates(index_col).merge(t, on=index_col)
    return df

## Inventory
`PedPanCancer\ ecDNA/restricted` doesn't have a SAMPLE_INFO file. Do I have that data somewhere?

RESULTS

- f3 (`SAMPLE_INFO PedPanCancer ecDNA.txt`) has all vcfs currently in the `PedPanCancer ecDNA/restricted` directory.  
- f1/f2 (`SAMPLE_INFO_2022-09-06.txt`) has all vcfs currently in the `PedPanCancer ecDNA/Data Requested 2022-09-06` directory.
- vcfs in `restricted` and `Data Requested 2022-09-06` are (nearly) mutually exclusive (6 gvcfs shared)

In [ ]:
r1=set(list(zip(*get_dx_files(ppce_id,gvcf_paths[0])))[0])
r2=set(list(zip(*get_dx_files(ppce_id,svcf_paths[0])))[0])

In [ ]:
for f in [f1,f2,f3,f4,f5]:
    g = set(get_vcfs(f).file_name)
    text_venn2(g,r1)
    print('-'*20)
    text_venn2(g,r2)
    print('='*20)

In [ ]:
r3=set(list(zip(*get_dx_files(ppce_id,gvcf_paths[1])))[0])
r4=set(list(zip(*get_dx_files(ppce_id,svcf_paths[1])))[0])

In [ ]:
for f in [f1,f2,f3,f4,f5]:
    g = set(get_vcfs(f).file_name)
    text_venn2(g,r3)
    print('-'*20)
    text_venn2(g,r4)
    print('='*20)

In [ ]:
text_venn2(r1,r3)
print('-'*20)
text_venn2(r2,r4)
print('='*20)

## Identify somatic variant files 

Download with:
```
mkdir -p out/somatic
while read fid; do
  echo "Downloading $fid"
  dx download "$fid" --output out/somatic
done < out/somatic_file_ids.txt
```

NOTES  
- Almost all samples in `restricted` were reanalyzed in `2022`, see eg. `SJACT001_D_G.Somatic.vcf.gz` in 2022-09-06 and `SJACT001_D.Somatic.vcf.gz` in restricted.
- Just use `2022`; this avoids the following problems:
    - f2 has extra columns f3 doesn't.  
    - `SAMPLE_INFO PedPanCancer ecDNA.txt` doesn't have file ids, probably need them.
    - deduplicating sample_ids
- Bad files:
    - file-Fy4yqfQ91321yGF656K44g5z, file-Fy4ybf091327bb3BG271XJ4x, SJST030131_D3_G1.Somatic.vcf.gz(.tbi)

In [ ]:
# Globals redefined here for clarity
biosamples = data_imports.import_biosamples()
f2=Path(PROJECT_ROOT,SAMPLE_INFO_DIR,'SAMPLE_INFO_2022-09-06.txt')
ppce_id='project-Fz9yBjQ9fj2QPbFY16B8JG8X'
svcf_path = '/Data Requested 2022-09-06/Somatic_VCF'
## Hack hack: remove corrupted somatic sequencing files using a blacklist
svcf_blacklist = [
    'file-Fy4yqfQ91321yGF656K44g5z', #SJST030131_D3_G1.Somatic.vcf.gz
    'file-Fy4ybf091327bb3BG271XJ4x', #SJST030131_D3_G1.Somatic.vcf.gz.tbi
]

def get_somatic_files(blacklist):
    '''
    Identify the somatic variant files of interest satisfying the following conditions:
    - Sample is in our cohort
    - File is in the 2022 subset
    '''
    def get_somatic_helper(sample_info,filenames, cohort_samples):
        df = sample_info
        accepted_sample_types = ['diagnosis','relapse','metastasis','autopsy']
        df = df[df.sample_type.isin(accepted_sample_types) & 
                (df.file_type == 'Somatic_VCF') &
                df.file_name.isin(filenames) &
                df.sample_name.isin(cohort_samples)
        ].copy()
        return df
        
    df = get_somatic_helper(read_sample_info(f2), # sample info for 2022-09-06
                            set(list(zip(*get_dx_files(ppce_id,svcf_path)))[0]), # files in 2022-09-06
                            biosamples.index
                           )
    df = flatten(df[~df.file_id.isin(blacklist)].copy(),'sample_name')
    return df


In [ ]:
somatic_data = get_somatic_files(svcf_blacklist)

In [ ]:
somatic_data

In [ ]:
with open('out/sj_somatic_file_ids.txt', 'w') as f:
    f.writelines(somatic_data.vcf_file_id + '\n')
    f.writelines(somatic_data.tbi_file_id + '\n')

In [ ]:
somatic_data.to_csv('out/sj_somatic_vcfs.tsv',sep='\t',index=False)

## Identify germline variant files

NOTES:  
- WXS and WGS available for the same patient in some cases.
- WXS only and not WGS available for 1% (13/1085) of patients, reasonable to exclude WXS.
- 2 patients with 2 germline WGS runs??
    - SJ001004 sequenced twice:
        - SJMEL001004_G1, file-FBvJ4fQ9KG5ZZFkV52q0GByb, 3.26.18, 4.95Gb,  
        - SJMEL001004_G2, file-FF672409pqyqzV1J24Qf4XGq, 4.10.18, 4.6Gb)  
        - Note G1 metadata (project, project id) in my SAMPLE_INFO doesn't seem to match what is currently on SJC.
    - SJ009003 sequencing analyzed twice? (file-FF67j4j994Pb46QvFB4X3X9F, 4.10.18, 5.65Gb  
                                           file-FBvQj309qP8Y15B73g9y59F2, 3.27.18, 6.29Gb)  
- Arbitrary solution - use most recent files.

In [ ]:
# Globals redefined here for clarity
patients = data_imports.import_patients()
f2=Path(PROJECT_ROOT,SAMPLE_INFO_DIR,'SAMPLE_INFO_2022-09-06.txt')
f3=Path(PROJECT_ROOT,SAMPLE_INFO_DIR,'SAMPLE_INFO PedPanCancer ecDNA.txt') # 15280 vcfs
ppce_id='project-Fz9yBjQ9fj2QPbFY16B8JG8X'
gvcf_paths = ['/restricted/gVCF/','/Data Requested 2022-09-06/gVCF/']
## Hack hack: remove duplicate germline sequencing files using a blacklist
gvcf_blacklist = [
    'file-FBvJ4fQ9KG5ZZFkV52q0GByb', #SJMEL001004_G1.WholeGenome.g.vcf.gz
    'file-FBvJ4p09KG5zzqGk522FB27j', #SJMEL001004_G1.WholeGenome.g.vcf.gz.tbi
    'file-FBvQj309qP8Y15B73g9y59F2', #SJHGG003_G.WholeGenome.g.vcf.gz, 3.27.18 version
    'file-FBvQj689qP8v9gbb3fYP6Bx1', #SJHGG003_G.WholeGenome.g.vcf.gz.tbi, 3.27.18 version
]

def get_germline_helper(sample_info, dx_files, cohort_patients):
    df = sample_info
    dx_files = list(zip(*dx_files))
    dx_files = pd.Series(index=dx_files[0], data=dx_files[1], name='file_id')
    df = df[(df.sample_type == 'germline') & 
            (df.file_type == 'gVCF') &
            (df.sequencing_type == 'WGS') &
            df.file_name.isin(dx_files.index) &
            df.subject_name.isin(cohort_patients)
    ].copy()
    if 'file_id' not in df.columns:
        df = df.join(dx_files,how='inner',on='file_name')
    # move .tbi to new column
    return df
    
def get_germline_files(blacklist):
    '''
    Identify the somatic variant files of interest satisfying the following conditions:
    - Sample is in our cohort
    - File is in the 2022 subset
    '''
    # 2022
    d1 = get_germline_helper(read_sample_info(f2), # sample info
                            get_dx_files(ppce_id,gvcf_paths[1]), # (file_name, file_id)
                            patients.index
                           )
    d1 = flatten(d1[~d1.file_id.isin(blacklist)].copy())
    # restricted
    d2 = get_germline_helper(read_sample_info(f3), # sample info
                            get_dx_files(ppce_id,gvcf_paths[0]), # (file_name, file_id)
                            patients.index
                           )
    d2 = flatten(d2[~d2.file_id.isin(blacklist)].copy(),'subject_name')

    df = pd.concat([d1,d2]).drop_duplicates('subject_name')
    return df

In [ ]:
germline_data = get_germline_files(gvcf_blacklist)

In [ ]:
germline_data.head(n=2)

In [ ]:
with open('out/sj_germline_file_ids.txt', 'w') as f:
    f.writelines(germline_data.vcf_file_id + '\n')
    f.writelines(germline_data.tbi_file_id + '\n')

In [ ]:
germline_data.to_csv('out/sj_germline_vcfs.tsv',sep='\t',index=False)

### Scratch

In [ ]:
q1=get_dx_files(ppce_id,gvcf_paths[0])
q3=get_dx_files(ppce_id,gvcf_paths[1])

In [ ]:
r1=set(list(zip(*q1))[0])
r3=set(list(zip(*q3))[0])

In [ ]:
s1=read_sample_info(f3)
s3=read_sample_info(f2)

In [ ]:
s3.head(n=1)

In [ ]:
t3=get_germline_helper(s3,q3,patients.index)
t1=get_germline_helper(s1,q1,patients.index)

In [ ]:
t3.head()

In [ ]:
# Which samples were germline sequenced more than once??
df = t1
df = df.sort_values(['subject_name', 'file_name']).reset_index(drop=True)
df['ct'] = df.groupby('subject_name').cumcount()
df[df.ct >= 2]

In [ ]:
t1[(t1.subject_name.isin(['SJ009003', 'SJ001004'])) &
    (t1.file_type == 'gVCF') &
    (t1.sample_type == 'germline') &
    (t1.sequencing_type == 'WGS')]

In [ ]:
# In either dataset, are patient genomes analyzed more than once?
t1['ct'] = t1.groupby('sample_name').cumcount()+1
t3['ct'] = t3.groupby('sample_name').cumcount()+1
# yes, WXS and WGS analyses available for some patients.
# Considering only WGS, no duplicates.
print(t1.ct.unique(),t3.ct.unique())

In [ ]:
# Are there cases where WXS only is available?
t1_wxs_pts = set(t1[t1.sequencing_type=='WES'].subject_name)
t1_wgs_pts = set(t1[t1.sequencing_type=='WGS'].subject_name)
print(len(t1_wxs_pts - t1_wgs_pts),'/',len(t1_wxs_pts | t1_wgs_pts))
# 19 of 729 in t1

t3_wxs_pts = set(t3[t3.sequencing_type=='WES'].subject_name)
t3_wgs_pts = set(t3[t3.sequencing_type=='WGS'].subject_name)
print(len(t3_wxs_pts - t3_wgs_pts),'/',len(t3_wxs_pts | t3_wgs_pts))
# 11 / 376 in t3

print(len((t1_wxs_pts | t3_wxs_pts) - (t1_wgs_pts | t3_wgs_pts)),'/',len((t1_wxs_pts | t3_wxs_pts) | (t1_wgs_pts | t3_wgs_pts)))
# 13 / 1085

In [ ]:
# How many duplicated across both datasets? 
print(len(t1_wgs_pts & t3_wgs_pts),'/',len(t1_wgs_pts | t3_wgs_pts))
# 3 / 1072
# are the data different? 
pts_of_interest = (t1_wgs_pts & t3_wgs_pts)
t1[t1.subject_name.isin(pts_of_interest) & (t1.file_name.str.endswith('gz'))]
# Yes
# SJ000101: sample_name SJHGG101_G -> SJHGG000101_G1
# SJ030020: sample_name SJMB030020_G1 -> SJHGG030020_G1
#           2022 metadata indicates patient had both HGG and MBL
#are the files different?

In [ ]:
t3[t3.subject_name.isin(pts_of_interest) & (t3.file_name.str.endswith('gz'))]
